In [1]:
import mocet
import os
import pickle
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from numpy.polynomial.legendre import Legendre
from sklearn.linear_model import LinearRegression

def make_poly_regressors(n_samples, order=2):
    X = np.ones((n_samples, 0))
    for d in range(order):
        poly = Legendre.basis(d + 1)
        poly_trend = poly(np.linspace(-1, 1, n_samples))
        X = np.hstack((X, poly_trend[:, None]))
    return X

def polynomial_detrending(pupil_data, polynomial_order):
    X = make_poly_regressors(len(pupil_data), order=polynomial_order)
    dedrift_regressor = np.zeros((len(pupil_data), 2))
    for i in range(2):
        reg = LinearRegression().fit(X, pupil_data[:, i])
        dedrift_regressor[:, i] = reg.predict(X)
    pupil_data = pupil_data[:, :2] - dedrift_regressor
    return pupil_data

subject_pool = {
                'sub-003':{'ses-07R':([1,2,3,4,5], False),
                           'ses-13R':([1,2,4,5,6], False)},
                'sub-004':{'ses-07R':([1,2,3,4,5,6], False),
                           'ses-13':([1,2,3,4,5,6], False)},
                'sub-005':{'ses-07':([1,2,3,4,5,6], True)},
                'sub-006':{'ses-07R':([1,2,3,4,5,6], False),
                           'ses-13':([1,2,3,4,5,6], False)},
                'sub-008':{'ses-07R':([2,3,4,5,6], False),
                           'ses-13':([1,2,3,4,5,6], False)},
                'sub-009':{'ses-07':([1,2,3,4,5,6], False),
                           'ses-13':([1,2,3,5,6], False)},
                'sub-010':{'ses-07':([1,2,3,4,5], False),
                           'ses-13':([1,2,3,4,5,6], False)},
                'sub-011':{'ses-07':([1,2,3,4,5,6], False),
                           'ses-13':([1,2,3,4,5,6], False)},
                'sub-012':{'ses-07':([1,2,4,5,6], False)},
                'sub-013':{'ses-07':([1,2,3,4], False)},
                'sub-014':{'ses-07':([2,3,4,5,6], False)},
                'sub-015':{'ses-07':([1,2,3,4,5,6], False),
                           'ses-13':([1,2,3,4,5,6], False)},
                'sub-016':{'ses-07':([1,2,3,4,5,6], False),
                           'ses-13':([1,2,3,4,5,6], False)},
                'sub-017':{'ses-07':([1,2,3,4,5,6], False),
                           'ses-13':([1,2,3,4,5], False)},
                'sub-018':{'ses-07':([1,2,3,4,5,6], False),
                           'ses-13':([1,2,3,4,5,6], False)},
                'sub-020':{'ses-07':([1,2,3,4,5,6], False),
                           'ses-13':([1,2,3,4,5,6], False)},
                'sub-021':{'ses-07':([1,2,4,5,6], False),
                           'ses-13':([1,2,4,5,6], False)},
                'sub-JJY':{'ses-07':([1,2,3,4,5,6], False)},
                'sub-KMY':{'ses-07':([1,2,3,4,5,6], False)},
                'sub-PJW':{'ses-07':([1,2,3,4,6], True)},
                'sub-PBJ':{'ses-07':([1,2,3,4,5], False)}
                }

history_onset = {'sub-005': [28.66, 29.32, 28.12, 33.7, 36.1, 27.46],
                 'sub-PJW': [35, 30.8, 28.66, 26.58, None, 27.42]}
task_duration = 816

calibration_onsets = [1, 494]
calibration_points = [24, 12]
interval = 1.6 
calibration_offset_start = 0.55
calibration_offset_end = -0.55
px_per_deg = 78.43
task = 'task-mcHERDING'
method = 'linear'
testable_data = pickle.load(open('../testable_data_list.pkl', 'rb'))

subjects = []
for key in list(testable_data.keys()):
    subjects.append(key[0])
subjects = list(set(subjects))
subjects.sort()

In [2]:
for subject in subjects:
    sessions = subject_pool[subject].keys()
    for session in sessions:
        runs, history_loss = subject_pool[subject][session]
        root = f'../../_DATA/{subject}/{session}'
        for r in runs:
            run = f'run-{r}'
            np.random.seed(0)
            key = (subject, session, task, run)
            if key in testable_data.keys():
                log_fname = f'{root}/{subject}_{session}_{task}_{run}_recording-eyetracking_physio_log.csv'
                data_fname = f'{root}/{subject}_{session}_{task}_{run}_recording-eyetracking_physio_dat.txt'
                confounds_fname = f'{root}/{subject}_{session}_{task}_{run}_desc-confounds_timeseries.tsv'
                
                if history_loss: 
                    start = history_onset[subject][r-1]
                else:
                    history_fname = f'{root}/{subject}_{session}_{task}_{run}_recording-eyetracking_physio_his.txt'
                    start, _, _ = mocet.utils.get_avotec_history(history_fname)
                
                # log, data, confound, start
                pupil_data, pupil_timestamps, pupil_confidence, pupil_diameter = mocet.utils.clean_avotec_data(log_fname,
                                                                                                        data_fname,
                                                                                                        start=start,
                                                                                                        duration=task_duration)
                pupil_data = mocet.apply_mocet(pupil_data, confounds_fname,
                                               large_motion_params=False ,
                                               polynomial_order=3)
                
                t_cal = testable_data[key][0]
                t_val = testable_data[key][2]
            
                offset = calibration_onsets[t_cal]
                calibration_pupils = []
                for i in np.arange(calibration_points[t_cal]):
                    start = (offset+i)*interval + calibration_offset_start
                    end = (offset+i+1)*interval + calibration_offset_end
                    log_effective = np.logical_and(pupil_timestamps >= start*1000, pupil_timestamps < end*1000)
                    calibration_pupils.append([np.nanmean(pupil_data[log_effective,0]),
                                              np.nanmean(pupil_data[log_effective,1])])
                calibration_pupils = np.array(calibration_pupils)
            
                repeat = True if calibration_points[t_cal]==24 else False
                calibrator = mocet.EyetrackingCalibration(repeat=repeat, method=method)
                calibrator.fit(calibration_pupils[:, 0], calibration_pupils[:, 1])
                gaze_coordinates = calibrator.transform(pupil_data)
                
                np.save(f'eyetracking_data/mocet/{subject}_{session}_{task}_{run}_gaze_coordinate.npy', gaze_coordinates)
                np.save(f'eyetracking_data/{subject}_{session}_{task}_{run}_gaze_timestamp.npy', pupil_timestamps)
                np.save(f'eyetracking_data/{subject}_{session}_{task}_{run}_pupil_diameter.npy', pupil_diameter)
                np.save(f'eyetracking_data/{subject}_{session}_{task}_{run}_pupil_confidence.npy', pupil_confidence)

In [7]:

def make_poly_regressors(n_samples, order=2):
    X = np.ones((n_samples, 0))
    for d in range(order):
        poly = Legendre.basis(d + 1)
        poly_trend = poly(np.linspace(-1, 1, n_samples))
        X = np.hstack((X, poly_trend[:, None]))
    return X

def polynomial_detrending(pupil_data, polynomial_order):
    X = make_poly_regressors(len(pupil_data), order=polynomial_order)
    dedrift_regressor = np.zeros((len(pupil_data), 2))
    for i in range(2):
        reg = LinearRegression().fit(X, pupil_data[:, i])
        dedrift_regressor[:, i] = reg.predict(X)
    pupil_data = pupil_data[:, :2] - dedrift_regressor
    return pupil_data

p = 3
for subject in subjects:
    sessions = subject_pool[subject].keys()
    for session in sessions:
        runs, history_loss = subject_pool[subject][session]
        root = f'../../_DATA/{subject}/{session}'
        for r in runs:
            run = f'run-{r}'
            np.random.seed(0)
            key = (subject, session, task, run)
            if key in testable_data.keys():
                
                log_fname = f'{root}/{subject}_{session}_{task}_{run}_recording-eyetracking_physio_log.csv'
                data_fname = f'{root}/{subject}_{session}_{task}_{run}_recording-eyetracking_physio_dat.txt'
                confounds_fname = f'{root}/{subject}_{session}_{task}_{run}_desc-confounds_timeseries.tsv'
                
                if history_loss: 
                    start = history_onset[subject][r-1]
                else:
                    history_fname = f'{root}/{subject}_{session}_{task}_{run}_recording-eyetracking_physio_his.txt'
                    start, _, _ = mocet.utils.get_avotec_history(history_fname)
                
                # log, data, confound, start
                pupil_data, pupil_timestamps, pupil_confidence, pupil_diameter = mocet.utils.clean_avotec_data(log_fname,
                                                                                                        data_fname,
                                                                                                        start=start,
                                                                                                        duration=task_duration)
                pupil_data = polynomial_detrending(pupil_data, polynomial_order=p)
                
                t_cal = testable_data[key][0]
                t_val = testable_data[key][2]
            
                offset = calibration_onsets[t_cal]
                calibration_pupils = []
                for i in np.arange(calibration_points[t_cal]):
                    start = (offset+i)*interval + calibration_offset_start
                    end = (offset+i+1)*interval + calibration_offset_end
                    log_effective = np.logical_and(pupil_timestamps >= start*1000, pupil_timestamps < end*1000)
                    calibration_pupils.append([np.nanmean(pupil_data[log_effective,0]),
                                              np.nanmean(pupil_data[log_effective,1])])
                calibration_pupils = np.array(calibration_pupils)
                
                repeat = True if calibration_points[t_cal]==24 else False
                calibrator = mocet.EyetrackingCalibration(repeat=repeat, method=method)
                calibrator.fit(calibration_pupils[:, 0], calibration_pupils[:, 1])
                gaze_coordinates = calibrator.transform(pupil_data)
                
                np.save(f'eyetracking_data/polynomial/{subject}_{session}_{task}_{run}_gaze_coordinate.npy', gaze_coordinates)
        

In [4]:

p = 1
for subject in subjects:
    sessions = subject_pool[subject].keys()
    for session in sessions:
        runs, history_loss = subject_pool[subject][session]
        root = f'../../_DATA/{subject}/{session}'
        for r in runs:
            run = f'run-{r}'
            np.random.seed(0)
            key = (subject, session, task, run)
            if key in testable_data.keys():
                
                log_fname = f'{root}/{subject}_{session}_{task}_{run}_recording-eyetracking_physio_log.csv'
                data_fname = f'{root}/{subject}_{session}_{task}_{run}_recording-eyetracking_physio_dat.txt'
                confounds_fname = f'{root}/{subject}_{session}_{task}_{run}_desc-confounds_timeseries.tsv'
                
                if history_loss: 
                    start = history_onset[subject][r-1]
                else:
                    history_fname = f'{root}/{subject}_{session}_{task}_{run}_recording-eyetracking_physio_his.txt'
                    start, _, _ = mocet.utils.get_avotec_history(history_fname)
                
                # log, data, confound, start
                pupil_data, pupil_timestamps, pupil_confidence, pupil_diameter = mocet.utils.clean_avotec_data(log_fname,
                                                                                                        data_fname,
                                                                                                        start=start,
                                                                                                        duration=task_duration)
                pupil_data = polynomial_detrending(pupil_data, polynomial_order=p)
                
                t_cal = testable_data[key][0]
                t_val = testable_data[key][2]
            
                offset = calibration_onsets[t_cal]
                calibration_pupils = []
                for i in np.arange(calibration_points[t_cal]):
                    start = (offset+i)*interval + calibration_offset_start
                    end = (offset+i+1)*interval + calibration_offset_end
                    log_effective = np.logical_and(pupil_timestamps >= start*1000, pupil_timestamps < end*1000)
                    calibration_pupils.append([np.nanmean(pupil_data[log_effective,0]),
                                              np.nanmean(pupil_data[log_effective,1])])
                calibration_pupils = np.array(calibration_pupils)
                
                repeat = True if calibration_points[t_cal]==24 else False
                calibrator = mocet.EyetrackingCalibration(repeat=repeat, method=method)
                calibrator.fit(calibration_pupils[:, 0], calibration_pupils[:, 1])
                gaze_coordinates = calibrator.transform(pupil_data)
                
                np.save(f'eyetracking_data/linear/{subject}_{session}_{task}_{run}_gaze_coordinate.npy', gaze_coordinates)
        

In [5]:

for subject in subjects:
    sessions = subject_pool[subject].keys()
    for session in sessions:
        runs, history_loss = subject_pool[subject][session]
        root = f'../../_DATA/{subject}/{session}'
        for r in runs:
            run = f'run-{r}'
            np.random.seed(0)
            key = (subject, session, task, run)
            if key in testable_data.keys():
                
                log_fname = f'{root}/{subject}_{session}_{task}_{run}_recording-eyetracking_physio_log.csv'
                data_fname = f'{root}/{subject}_{session}_{task}_{run}_recording-eyetracking_physio_dat.txt'
                confounds_fname = f'{root}/{subject}_{session}_{task}_{run}_desc-confounds_timeseries.tsv'
                
                if history_loss: 
                    start = history_onset[subject][r-1]
                else:
                    history_fname = f'{root}/{subject}_{session}_{task}_{run}_recording-eyetracking_physio_his.txt'
                    start, _, _ = mocet.utils.get_avotec_history(history_fname)
                
                # log, data, confound, start
                pupil_data, pupil_timestamps, pupil_confidence, pupil_diameter = mocet.utils.clean_avotec_data(log_fname,
                                                                                                        data_fname,
                                                                                                        start=start,
                                                                                                        duration=task_duration)
                t_cal = testable_data[key][0]
                t_val = testable_data[key][2]
            
                offset = calibration_onsets[t_cal]
                calibration_pupils = []
                for i in np.arange(calibration_points[t_cal]):
                    start = (offset+i)*interval + calibration_offset_start
                    end = (offset+i+1)*interval + calibration_offset_end
                    log_effective = np.logical_and(pupil_timestamps >= start*1000, pupil_timestamps < end*1000)
                    calibration_pupils.append([np.nanmean(pupil_data[log_effective,0]),
                                              np.nanmean(pupil_data[log_effective,1])])
                calibration_pupils = np.array(calibration_pupils)
                
                repeat = True if calibration_points[t_cal]==24 else False
                calibrator = mocet.EyetrackingCalibration(repeat=repeat, method=method)
                calibrator.fit(calibration_pupils[:, 0], calibration_pupils[:, 1])
                gaze_coordinates = calibrator.transform(pupil_data)
                
                np.save(f'eyetracking_data/uncorrected/{subject}_{session}_{task}_{run}_gaze_coordinate.npy', gaze_coordinates)
        